# AksharDrishti — Kaggle Setup Notebook

**Notebook name:** `akshardrishti-train`  
**Accelerator:** GPU T4 x2 — set in *Settings → Accelerator*  
**Internet:** ON — set in *Settings → Internet*  
**Persistence:** Files only — set in *Settings → Persistence*  

Run cells top-to-bottom in order. Each cell prints its own PASS/FAIL so you can stop immediately if something is wrong rather than discovering the problem two hours into training.

**This notebook does NOT start any training.** It only sets up the environment.

---
## STEP 1 — GPU sanity check

In [ ]:
import subprocess, sys, os

print('=== Python ===')
print(sys.version)

print('\n=== nvidia-smi ===')
try:
    print(subprocess.check_output(['nvidia-smi'], stderr=subprocess.STDOUT).decode())
except Exception as e:
    print('ERROR: no GPU detected:', e)
    print('Go to Settings → Accelerator → GPU T4 x2, then re-run.')

print('\n=== PyTorch GPU check ===')
import torch
print('torch version  :', torch.__version__)
print('CUDA available :', torch.cuda.is_available())
print('Device count   :', torch.cuda.device_count())
for i in range(torch.cuda.device_count()):
    print(f'  GPU {i}: {torch.cuda.get_device_name(i)}')

assert torch.cuda.is_available(), 'STOP — no CUDA. Enable GPU T4 x2 in Settings.'

---
## STEP 2 — Clone repo and check out exact commit `594e7ff`

In [ ]:
REPO   = 'https://github.com/Ayush-04-spec/akshardrishti.git'
COMMIT = '594e7ff'
TARGET = '/kaggle/working/aksharDrishti'

import os, subprocess

def run(cmd, **kw):
    """Run a shell command, print output, raise on non-zero exit."""
    result = subprocess.run(cmd, shell=True, capture_output=True, text=True, **kw)
    if result.stdout.strip():
        print(result.stdout.strip())
    if result.stderr.strip():
        print(result.stderr.strip())
    result.check_returncode()
    return result.stdout.strip()

if not os.path.isdir(TARGET):
    print(f'Cloning {REPO} ...')
    run(f'git clone {REPO} {TARGET}')
else:
    print(f'Repo already exists at {TARGET}, skipping clone.')

print(f'\nChecking out commit {COMMIT} ...')
run(f'git -C {TARGET} checkout {COMMIT}')

actual = run(f'git -C {TARGET} rev-parse HEAD')
print(f'\nHEAD is now: {actual}')
assert actual.startswith(COMMIT), f'STOP — HEAD {actual!r} does not start with {COMMIT!r}'
print(f'PASS — commit matches {COMMIT}')

# Add repo to Python path for all subsequent cells
import sys
if TARGET not in sys.path:
    sys.path.insert(0, TARGET)
os.chdir(TARGET)
print(f'cwd → {os.getcwd()}')

---
## STEP 3 — Verify attached datasets

Before running this cell, make sure both datasets are attached as notebook inputs:
- **Add Data** (top-right panel) → search `akshardrishti-indicdlp-subset` → Add  
- **Add Data** → search `mozhi-dataset` → Add

In [ ]:
import os

INDICDLP_ROOT = '/kaggle/input/datasets/ayushshirsath03/akshardrishti-indicdlp-subset/indicdlp_subset'
MOZHI_ROOT    = '/kaggle/input/datasets/ayushshirsath03/mozhi-dataset'

# ── IndicDLP ────────────────────────────────────────────────────────────────
print('='*60)
print('IndicDLP subset:', INDICDLP_ROOT)
print('='*60)

assert os.path.isdir(INDICDLP_ROOT), (
    f'STOP — {INDICDLP_ROOT} not found.\n'
    'Add akshardrishti-indicdlp-subset as a notebook input (Add Data panel).'
)

indicdlp_top = sorted(os.listdir(INDICDLP_ROOT))
print('Top-level contents:')
for name in indicdlp_top:
    full = os.path.join(INDICDLP_ROOT, name)
    if os.path.isdir(full):
        n = sum(len(files) for _, _, files in os.walk(full))
        print(f'  {name}/  ({n:,} files)')
    else:
        print(f'  {name}  ({os.path.getsize(full):,} bytes)')

# Critical check: images/ and labels/ must exist and be non-empty
images_dir = os.path.join(INDICDLP_ROOT, 'images', 'train')
labels_dir = os.path.join(INDICDLP_ROOT, 'labels', 'train')

assert os.path.isdir(images_dir), (
    f'STOP — images/train/ missing from IndicDLP dataset.\n'
    f'The upload is incomplete (only metadata was uploaded).\n'
    f'Re-upload indicdlp_subset.zip via the Kaggle web interface.'
)
assert os.path.isdir(labels_dir), (
    f'STOP — labels/train/ missing from IndicDLP dataset.\n'
    f'Re-upload indicdlp_subset.zip via the Kaggle web interface.'
)

n_images = len(os.listdir(images_dir))
n_labels = len(os.listdir(labels_dir))
print(f'\nimages/train/: {n_images:,} files')
print(f'labels/train/: {n_labels:,} files')
assert n_images >= 9000, f'STOP — only {n_images} images, expected ~9,306. Incomplete upload.'
assert n_labels >= 9000, f'STOP — only {n_labels} labels, expected ~9,306. Incomplete upload.'
print(f'PASS — IndicDLP: {n_images:,} images, {n_labels:,} labels')

# ── Mozhi ───────────────────────────────────────────────────────────────────
print()
print('='*60)
print('Mozhi raw:', MOZHI_ROOT)
print('='*60)

assert os.path.isdir(MOZHI_ROOT), (
    f'STOP — {MOZHI_ROOT} not found.\n'
    'Add akshardrishti-mozhi-raw (mozhi-dataset) as a notebook input (Add Data panel).'
)

mozhi_top = sorted(os.listdir(MOZHI_ROOT))
print('Top-level contents:')
for name in mozhi_top:
    full = os.path.join(MOZHI_ROOT, name)
    if os.path.isdir(full):
        n = sum(len(files) for _, _, files in os.walk(full))
        print(f'  {name}/  ({n:,} files)')
    else:
        print(f'  {name}  ({os.path.getsize(full):,} bytes)')

# Critical check: hindi/ and marathi/ image folders must exist
for lang in ('hindi', 'marathi'):
    for split in ('train', 'val', 'test'):
        d = os.path.join(MOZHI_ROOT, lang, split, 'images')
        assert os.path.isdir(d), (
            f'STOP — {d} not found.\n'
            f'The mozhi_raw.zip upload may be incomplete or the ZIP used backslash paths.\n'
            f'Re-upload using the Python-generated mozhi_raw.zip (forward-slash paths).'
        )
        n = len(os.listdir(d))
        print(f'  {lang}/{split}/images/: {n:,} files')

# Check index CSVs are present
for csv_name in ('index_train.csv', 'index_val.csv', 'index_test.csv', 'charset_deva.txt'):
    path = os.path.join(MOZHI_ROOT, csv_name)
    assert os.path.isfile(path), f'STOP — {csv_name} missing from Mozhi dataset.'
    print(f'  {csv_name}: {os.path.getsize(path):,} bytes  ✓')

print(f'\nPASS — both datasets look complete.')

---
## STEP 4 — Install dependencies

Kaggle's base image already has `torch`, `torchvision`, `numpy`, `opencv`, and `pandas`.  
We record what's already there **before** installing so the diff is clear.

In [ ]:
import subprocess

PKGS_TO_CHECK = 'torch|torchvision|ultralytics|sahi|pydantic|transformers|datasets|huggingface'

def pip_grep(pattern):
    out = subprocess.run(
        f'pip list 2>/dev/null | grep -iE "{pattern}"',
        shell=True, capture_output=True, text=True
    ).stdout.strip()
    return out if out else '(none)'

print('=== Package versions BEFORE install ===')
print(pip_grep(PKGS_TO_CHECK))

In [ ]:
# Install ONLY what Kaggle's base image doesn't already provide.
# torch / torchvision / numpy / opencv are pre-installed — do NOT force-reinstall them.

print('Installing ultralytics, sahi, pydantic, transformers, datasets, jiwer ...')
!pip install -q \
    'ultralytics>=8.3.0' \
    'sahi>=0.11.15' \
    'pydantic>=2.6' \
    'transformers>=4.40' \
    'datasets>=2.18' \
    'huggingface-hub>=0.22' \
    'jiwer>=3.0' \
    'wandb>=0.16' \
    'pymupdf'

print('\nInstalling Tesseract system package + language packs ...')
!apt-get install -qq -y \
    tesseract-ocr \
    tesseract-ocr-hin \
    tesseract-ocr-mar \
    fonts-indic \
    fonts-noto-core \
    > /dev/null 2>&1

print('Installing pytesseract ...')
!pip install -q pytesseract

print('\nDone.')

In [ ]:
print('=== Package versions AFTER install ===')
print(pip_grep(PKGS_TO_CHECK))

# Hard assertions so training cells fail fast if a key package is missing
import importlib
required = {
    'ultralytics': 'ultralytics',
    'sahi':        'sahi',
    'pydantic':    'pydantic',
    'transformers':'transformers',
    'pytesseract': 'pytesseract',
}
all_ok = True
for display_name, module_name in required.items():
    try:
        mod = importlib.import_module(module_name)
        ver = getattr(mod, '__version__', 'unknown')
        print(f'  {display_name:<14} {ver}  ✓')
    except ImportError:
        print(f'  {display_name:<14} MISSING  ✗')
        all_ok = False

assert all_ok, 'STOP — one or more required packages failed to install. See ✗ above.'
print('\nPASS — all required packages installed.')

In [ ]:
# Smoke-test: import the akshardrishti package itself
import sys, os
TARGET = '/kaggle/working/aksharDrishti'
if TARGET not in sys.path:
    sys.path.insert(0, TARGET)
os.chdir(TARGET)

import akshardrishti
print(f'akshardrishti version: {akshardrishti.__version__}')

from akshardrishti.config import Config, ClassMap
from akshardrishti.schema import Document, Page, Region, RegionType, BBox
from akshardrishti.recognize.crnn_model import CRNN, Charset

cfg = Config.load('configs/pipeline.yaml')
cm  = ClassMap.load('configs/class_map.yaml')
print(f'Config loaded  — hash: {cfg.hash()}')
print(f'ClassMap loaded — {len(cm.targets)} target classes: {cm.targets}')
print('\nPASS — package imports OK.')

---
## STEP 5 — Fix Mozhi CSV paths

The index CSVs contain absolute Windows paths from the machine where they were generated.  
This step rewrites them to `/kaggle/input/datasets/ayushshirsath03/mozhi-dataset/...` and saves the fixed  
versions to `/kaggle/working/mozhi_fixed/` (writable, unlike `/kaggle/input/`).

In [ ]:
!python /kaggle/working/aksharDrishti/scripts/fix_mozhi_paths_kaggle.py \
    --input-dir  /kaggle/input/datasets/ayushshirsath03/mozhi-dataset \
    --output-dir /kaggle/working/mozhi_fixed \
    --spot-check-n 5

In [ ]:
# Confirm the fixed CSVs exist and spot-check 5 rows resolve to real files
import os, csv, random

FIXED_DIR = '/kaggle/working/mozhi_fixed'

for csv_name in ('index_train.csv', 'index_val.csv', 'index_test.csv'):
    path = os.path.join(FIXED_DIR, csv_name)
    assert os.path.isfile(path), f'STOP — {path} not found. Did the fix script run successfully?'
    with open(path, encoding='utf-8') as f:
        rows = list(csv.DictReader(f))
    print(f'{csv_name}: {len(rows):,} rows')

print()

# Extended spot-check: 5 random rows from train
train_csv = os.path.join(FIXED_DIR, 'index_train.csv')
with open(train_csv, encoding='utf-8') as f:
    all_rows = list(csv.DictReader(f))

rng = random.Random(99)
sample = rng.sample(all_rows, 5)

print('Spot-check (5 random rows from index_train.csv):')
all_exist = True
for row in sample:
    exists = os.path.exists(row['image_path'])
    all_exist = all_exist and exists
    mark = 'OK' if exists else 'MISSING'
    print(f'  [{mark}]  {row["language"]:<8}  {row["text"]:<20}  {row["image_path"]}')

assert all_exist, (
    'STOP — some image files are missing.\n'
    'The Mozhi dataset upload may be incomplete. '
    'Verify akshardrishti-mozhi-raw on Kaggle shows ~566 MB.'
)
print('\nPASS — all 5 sampled files exist on disk.')

---
## Setup complete — summary

All five steps passed. Record the key values below for the lab report / paper.

In [ ]:
import subprocess, torch, importlib, os

TARGET = '/kaggle/working/aksharDrishti'

commit = subprocess.run(
    f'git -C {TARGET} rev-parse HEAD',
    shell=True, capture_output=True, text=True
).stdout.strip()

def ver(mod_name):
    try:
        return importlib.import_module(mod_name).__version__
    except Exception:
        return 'n/a'

print('=' * 56)
print('  AksharDrishti — Setup Summary')
print('=' * 56)
print(f'  Commit          : {commit}')
print(f'  PyTorch         : {torch.__version__}')
print(f'  CUDA            : {torch.version.cuda}')
print(f'  GPU count       : {torch.cuda.device_count()}')
print(f'  ultralytics     : {ver("ultralytics")}')
print(f'  sahi            : {ver("sahi")}')
print(f'  pydantic        : {ver("pydantic")}')
print(f'  transformers    : {ver("transformers")}')
print(f'  pytesseract     : {ver("pytesseract")}')
print(f'  IndicDLP images : {len(os.listdir("/kaggle/input/datasets/ayushshirsath03/akshardrishti-indicdlp-subset/indicdlp_subset/images/train")):,}')
print(f'  IndicDLP labels : {len(os.listdir("/kaggle/input/datasets/ayushshirsath03/akshardrishti-indicdlp-subset/indicdlp_subset/labels/train")):,}')
print(f'  Mozhi fixed CSVs: /kaggle/working/mozhi_fixed/')
print('=' * 56)
print('  STATUS: READY FOR TRAINING')
print('=' * 56)

---
---
# LAYOUT DETECTOR TRAINING

Cells below train **Model \#1 — YOLO11l layout detector** on the IndicDLP subset.  
Run in order. The smoke test (LAYOUT STEP 3) must PASS before launching the full run (LAYOUT STEP 4).

---
## LAYOUT STEP 1 — Fix data.yaml (blocking — run before any training)

In [ ]:
import yaml, os

INDICDLP_ROOT = '/kaggle/input/datasets/ayushshirsath03/akshardrishti-indicdlp-subset/indicdlp_subset'
SRC_YAML      = os.path.join(INDICDLP_ROOT, 'data.yaml')
DST_YAML      = '/kaggle/working/data_kaggle.yaml'

assert os.path.isfile(SRC_YAML), f'STOP — source data.yaml not found at {SRC_YAML}'

print(f'Original data.yaml contents ({SRC_YAML}):')
print(open(SRC_YAML).read())
print('---')

with open(SRC_YAML) as f:
    cfg = yaml.safe_load(f)

# Override: point path to the Kaggle mount location
cfg['path']  = INDICDLP_ROOT
cfg['train'] = 'images/train'

# Remove val/test — this subset only has a train split
cfg.pop('val',  None)
cfg.pop('test', None)

with open(DST_YAML, 'w') as f:
    yaml.dump(cfg, f, allow_unicode=True, sort_keys=False)

written = open(DST_YAML).read()
print(f'Fixed data.yaml written to {DST_YAML}:')
print(written)

# Sanity checks: no Windows paths remain
assert 'AksharDrishti' not in written, 'STOP — Windows path still present in data.yaml!'
assert 'D:\\\\' not in written,        'STOP — Windows drive letter still present!'
assert os.path.isdir(os.path.join(INDICDLP_ROOT, 'images', 'train')), \
    f'STOP — images/train/ not found under {INDICDLP_ROOT}'

n_imgs = len(os.listdir(os.path.join(INDICDLP_ROOT, 'images', 'train')))
n_lbls = len(os.listdir(os.path.join(INDICDLP_ROOT, 'labels', 'train')))
print(f'images/train/: {n_imgs:,} files')
print(f'labels/train/: {n_lbls:,} files')
print('PASS — data_kaggle.yaml is clean and image/label directories exist.')

---
## LAYOUT STEP 2 — Per-epoch checkpoint backup (background polling thread)

**Mechanism used: background polling thread — NOT an ultralytics callback.**

Reason: `train_layout.py` launches training via a blocking `model.train()` call inside
a subprocess (`subprocess.Popen`). Ultralytics callbacks (e.g. `on_train_epoch_end`)
must be registered on the YOLO Python object *before* calling `.train()` — they are
inaccessible from outside a subprocess. A daemon thread polling the checkpoint directory
on disk every 60 seconds achieves the same effect without any changes to train_layout.py.

In [ ]:
import os, shutil, time, threading
from pathlib import Path

# Ultralytics saves checkpoints here (matches --project / --name in training command)
CKPT_SRC = Path('/kaggle/working/aksharDrishti/runs/detect/akshardrishti_layout/weights')
CKPT_DST = Path('/kaggle/working/weights/layout')
CKPT_DST.mkdir(parents=True, exist_ok=True)

POLL_INTERVAL = 60  # seconds

_stop_backup = threading.Event()

def _backup_loop():
    """Copy best.pt and last.pt whenever they are newer than our cached copy."""
    seen_mtime = {}
    while not _stop_backup.is_set():
        if CKPT_SRC.exists():
            for name in ('best.pt', 'last.pt'):
                src = CKPT_SRC / name
                dst = CKPT_DST / name
                if src.exists():
                    mtime = src.stat().st_mtime
                    if seen_mtime.get(name, 0) < mtime:
                        try:
                            shutil.copy2(src, dst)
                            seen_mtime[name] = mtime
                            print(f'  [{time.strftime("%H:%M:%S")}] backed up {name} → {dst}',
                                  flush=True)
                        except Exception as e:
                            print(f'  backup warning: {e}', flush=True)
        _stop_backup.wait(POLL_INTERVAL)

_backup_thread = threading.Thread(target=_backup_loop, daemon=True, name='ckpt-backup')
_backup_thread.start()

print(f'Checkpoint backup thread started (interval: {POLL_INTERVAL}s).')
print(f'  Watching : {CKPT_SRC}')
print(f'  Copying to: {CKPT_DST}')
print(f'  Thread alive: {_backup_thread.is_alive()}')
print()
print('Thread runs for the lifetime of this kernel. Stop manually with: _stop_backup.set()')

---
## LAYOUT STEP 3 — Smoke test (1 epoch on real data)

Runs `train_layout.py` for exactly **1 epoch** to confirm no path / shape / OOM errors
before committing 4–6 hours to the full run.

**Read the tail output carefully. If you see any error or non-zero exit code, stop here
and do NOT run LAYOUT STEP 4.**

In [ ]:
import subprocess, sys, os, time
from pathlib import Path

REPO    = '/kaggle/working/aksharDrishti'
LOG_DIR = '/kaggle/working/logs'
os.makedirs(LOG_DIR, exist_ok=True)

SMOKE_CMD = [
    sys.executable,
    f'{REPO}/train/train_layout.py',
    '--data',       '/kaggle/working/data_kaggle.yaml',
    '--model',      'yolo11l.pt',
    '--epochs',     '1',
    '--batch',      '16',
    '--imgsz',      '1024',
    '--device',     '0,1',
    '--name',       'akshardrishti_layout',
    '--backup-dir', '/kaggle/working/weights/layout',
]

print('=== SMOKE TEST — 1 epoch ===')
print('Command:', ' '.join(SMOKE_CMD))
print()

t0 = time.time()
result = subprocess.run(
    SMOKE_CMD,
    cwd=REPO,
    capture_output=True,
    text=True,
    env={**os.environ, 'WANDB_MODE': 'disabled'},
)
elapsed = time.time() - t0

combined = (result.stdout + result.stderr).strip()
lines    = combined.splitlines()

print(f'--- Output tail (last 40 of {len(lines)} lines) ---')
print('\n'.join(lines[-40:]))
print('--- end ---')
print()
print(f'Exit code : {result.returncode}')
print(f'Duration  : {elapsed:.0f}s')

# Save full log
smoke_log = f'{LOG_DIR}/smoke_test.log'
with open(smoke_log, 'w') as f:
    f.write(combined)
print(f'Full log  : {smoke_log}')
print()

if result.returncode != 0:
    print('=' * 56)
    print(f'  SMOKE TEST FAILED  (exit code {result.returncode})')
    print('  DO NOT run LAYOUT STEP 4 until this error is fixed.')
    print('=' * 56)
    raise SystemExit('Smoke test failed — see output above.')

# Confirm checkpoint was created
last_pt = Path(f'{REPO}/runs/detect/akshardrishti_layout/weights/last.pt')
if last_pt.exists():
    size_mb = last_pt.stat().st_size / 1024 / 1024
    print(f'Checkpoint: {last_pt}  ({size_mb:.1f} MB)  ✓')
else:
    print(f'WARNING: expected checkpoint not found at {last_pt}')

print()
print('=' * 56)
print('  SMOKE TEST PASSED')
print('  Proceed to LAYOUT STEP 4 to launch the full 60-epoch run.')
print('=' * 56)

---
## LAYOUT STEP 4 — Full 60-epoch run (background process)

**Only run this cell if LAYOUT STEP 3 printed `SMOKE TEST PASSED`.**

Uses `--resume` to continue from the epoch-1 smoke-test checkpoint (no wasted work).
The cell returns immediately; training runs in the background.
stdout/stderr are logged to `/kaggle/working/logs/layout_train.log`.

In [ ]:
import subprocess, sys, os, time
from pathlib import Path

REPO     = '/kaggle/working/aksharDrishti'
LOG_FILE = '/kaggle/working/logs/layout_train.log'
os.makedirs('/kaggle/working/logs', exist_ok=True)

# Guard: require smoke-test checkpoint
smoke_ckpt = Path(f'{REPO}/runs/detect/akshardrishti_layout/weights/last.pt')
assert smoke_ckpt.exists(), (
    f'STOP — smoke-test checkpoint not found at {smoke_ckpt}.\n'
    'Run LAYOUT STEP 3 first and confirm it prints SMOKE TEST PASSED.'
)

TRAIN_CMD = [
    sys.executable,
    f'{REPO}/train/train_layout.py',
    '--data',       '/kaggle/working/data_kaggle.yaml',
    '--model',      'yolo11l.pt',
    '--epochs',     '60',
    '--batch',      '16',
    '--imgsz',      '1024',
    '--device',     '0,1',
    '--name',       'akshardrishti_layout',
    '--backup-dir', '/kaggle/working/weights/layout',
    '--resume',
]

print('Launching full 60-epoch run in background...')
print('Command:', ' '.join(TRAIN_CMD))
print(f'Log: {LOG_FILE}')
print()

log_fh = open(LOG_FILE, 'w', buffering=1)
proc = subprocess.Popen(
    TRAIN_CMD,
    cwd=REPO,
    stdout=log_fh,
    stderr=subprocess.STDOUT,
    env={**os.environ, 'WANDB_MODE': 'disabled'},
)
print(f'Process started. PID = {proc.pid}')
print()

# Wait 2 minutes then report status
print('Waiting 2 minutes for warm-up...')
for i in range(12):
    time.sleep(10)
    log_bytes = os.path.getsize(LOG_FILE) if os.path.exists(LOG_FILE) else 0
    alive = proc.poll() is None
    print(f'  {(i+1)*10:3d}s  log={log_bytes:,}B  '
          f'process={"running" if alive else f"EXITED code={proc.returncode}"}',
          flush=True)
    if not alive:
        break

print()
print('--- Last 15 lines of log ---')
with open(LOG_FILE) as f:
    tail = f.read().splitlines()
print('\n'.join(tail[-15:]))
print('---')
print()

if proc.poll() is not None:
    print(f'ERROR: process exited early (code {proc.returncode}). Check {LOG_FILE}')
else:
    print(f'PID {proc.pid} is running.')
    print()
    smi = subprocess.run(['nvidia-smi'], capture_output=True, text=True)
    print('--- nvidia-smi ---')
    print(smi.stdout)
    print('---')
    print()
    print('Monitor: !tail -f /kaggle/working/logs/layout_train.log')
    print('Backup thread is copying best.pt / last.pt every 60s to /kaggle/working/weights/layout/')
    print('Stop backup when done: _stop_backup.set()')

---
## LAYOUT STEP 5 — Validation results

### ⚠️  DO NOT RUN THIS CELL UNTIL TRAINING IS COMPLETE

Training is done when:
- `proc.poll() is not None` returns `True`, OR
- The last line of `/kaggle/working/logs/layout_train.log` contains the metrics table

**Checkpoint saved by train_layout.py as:**  
`/kaggle/working/weights/layout/akshardrishti_layout_best.pt`  
(The `--backup-dir` arg names it `{--name}_best.pt` — confirmed from the script source.)

In [ ]:
# ============================================================
# RUN AFTER TRAINING COMPLETES — DO NOT RUN DURING TRAINING
# ============================================================
import json, os, sys, csv
from pathlib import Path

REPO       = '/kaggle/working/aksharDrishti'
RUN_DIR    = Path(f'{REPO}/runs/detect/akshardrishti_layout')
BACKUP_DIR = Path('/kaggle/working/weights/layout')

# train_layout.py writes: {project}/{name}/akshardrishti_metrics.json
# and copies best.pt as:  {backup_dir}/{name}_best.pt
BEST_PT      = BACKUP_DIR / 'akshardrishti_layout_best.pt'
METRICS_JSON = RUN_DIR / 'akshardrishti_metrics.json'
RESULTS_CSV  = RUN_DIR / 'results.csv'

print('=== LAYOUT DETECTOR — FINAL RESULTS ===')
print()

# ── Option 1: read metrics JSON written by train_layout.py (preferred) ────
if METRICS_JSON.exists():
    r = json.loads(METRICS_JSON.read_text())
    print(f'Source: {METRICS_JSON}')
    print()
    print(f'  mAP@50-95 : {r["mAP50-95"]:.4f}   ← headline for Table 6.1')
    print(f'  mAP@50    : {r["mAP50"]:.4f}')
    print(f'  mAP@75    : {r["mAP75"]:.4f}')
    print(f'  precision : {r["precision"]:.4f}')
    print(f'  recall    : {r["recall"]:.4f}')
    if 'per_class' in r:
        print()
        print('  Per-class mAP@50-95 (sorted best → worst):')
        for cls, val in sorted(r['per_class'].items(), key=lambda x: -x[1]):
            bar = '█' * int(val * 30)
            print(f'    {cls:<14} {val:.4f}  {bar}')

# ── Option 2: re-run val on saved checkpoint if JSON missing ──────────────
elif BEST_PT.exists():
    print(f'Metrics JSON not found — re-running val on {BEST_PT} ...')
    sys.path.insert(0, REPO)
    from ultralytics import YOLO
    model   = YOLO(str(BEST_PT))
    metrics = model.val(data='/kaggle/working/data_kaggle.yaml', imgsz=1024, device='0,1')
    box = metrics.box
    print(f'  mAP@50-95 : {box.map:.4f}   ← headline for Table 6.1')
    print(f'  mAP@50    : {box.map50:.4f}')
    print(f'  mAP@75    : {box.map75:.4f}')
    print(f'  precision : {box.mp:.4f}')
    print(f'  recall    : {box.mr:.4f}')
    names = metrics.names
    if names and box.maps is not None:
        print()
        print('  Per-class mAP@50-95 (sorted best → worst):')
        rows = [(str(names[i] if isinstance(names,dict) else names[i]), float(v))
                for i, v in enumerate(box.maps)]
        for cls, val in sorted(rows, key=lambda x: -x[1]):
            bar = '█' * int(val * 30)
            print(f'    {cls:<14} {val:.4f}  {bar}')

else:
    print('STOP — no checkpoint found.')
    print(f'  Expected : {BEST_PT}')
    print(f'  Also try : {RUN_DIR}/weights/best.pt')
    print('Training may not have completed yet — check the log.')

# ── results.csv last row (sanity check) ───────────────────────────────────
if RESULTS_CSV.exists():
    rows = list(csv.DictReader(open(RESULTS_CSV)))
    if rows:
        last = rows[-1]
        print()
        print(f'  results.csv — epoch {last.get("                  epoch", last.get("epoch", "?")).strip()}:')
        for k, v in last.items():
            k = k.strip()
            if any(x in k.lower() for x in ('map', 'precision', 'recall', 'loss')):
                print(f'    {k:<40} {v.strip()}')

print()
print(f'Best weights: {BEST_PT}')
print('Copy mAP@50-95 into Table 6.1 of the report.')